# Lesson 07 - Planning Design Pattern

This notebook demonstrates the **Planning Design Pattern** for AI agents using the Microsoft Agent Framework.
You will learn how to break complex travel requests into structured subtasks, assign them to specialist agents,
and execute the resulting plan — all using structured output powered by Pydantic models.


## Kurulum


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Task Decomposition

Task decomposition is the core of the planning design pattern. Instead of asking a single agent to
handle a complex request end-to-end, we break the problem into smaller, well-defined **subtasks**.
Each subtask is assigned to a specialist agent (e.g., flights, hotels, activities) with clear
priorities and dependency ordering.

This approach provides several benefits:
- **Clarity**: each subtask has a single responsibility.
- **Parallelism**: independent subtasks can run concurrently.
- **Reliability**: failures are isolated to individual subtasks.
- **Budget tracking**: costs are estimated per subtask and rolled up.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Yapılandırılmış Çıktıyla Bir Planlama Ajanı Oluşturma

Planlama ajanı bir **ön büro koordinatörü** olarak görev yapar. Yüksek seviyeli bir seyahat talebi verildiğinde,
talebi alt görevlere ayıran, öncelikleri belirleyen ve bağımlılıkları tanımlayan yapılandırılmış bir `TravelPlan` oluşturur,
böylece bir konsiyerj veya yürütme katmanı işi gerçekleştirebilir.


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Uzman Araçlarla Bir Planı Yürütme

Ön büro görevlisi yapılandırılmış bir plan oluşturduktan sonra, **concierge ajanı** bunu uygular.
Her uzman araç bir alt görev kategorisini (uçuşlar, oteller, aktiviteler) yönetir. Concierge
bağımlılık sırasına göre planın alt görevleri arasında iterasyon yapar ve her birini
uygun araca gönderir.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Özet

Bu derste yapay zeka ajanları için **Planlama Tasarım Deseni**ni öğrendiniz:

1. **Görev Ayrıştırması** — Bir ön büro planlama ajanı karmaşık bir seyahat talebini  
   Pydantic modelleri kullanarak yapılandırılmış alt görevlere böler ve her birini öncelikler  
   ve bağımlılıklarla birlikte bir uzman ajana atar.  
2. **Yapılandırılmış Çıktı** — `response_format` iletildiğinde ajan, serbest biçimli metin yerine  
   doğrulanmış bir `TravelPlan` nesnesi döndürür, bu da sonraki işlemleri güvenilir kılar.  
3. **Plan Uygulaması** — Bir konsiyerj ajan, uzman araçları kullanarak alt görevleri iteratif şekilde  
   (`book_flight`, `reserve_hotel`, `book_activity`) planı uygular ve sonuçları raporlar.  

Bu desen, *ne yapılacağını* (planlama) *nasıl yapılacağını* (uygulama) ayırarak ajanları  
daha modüler, test edilebilir ve genişletmesi kolay hale getirir.  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalardan veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
